# Parte 4 — Duas bases

| Base | Uso |
|---|---|
| `base_analitica` | cópia completa de `df_enriquecido` — EDA, gráficos e Power BI |
| `base_modelavel` | só variáveis conhecidas **antes** do acidente + `acidente_fatal` |


## 4.0 Base enriquecida (refaz as Partes 2 e 3)

Notebook autônomo: reconstrói `df_enriquecido` para rodar isolado.


In [1]:
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd


def localizar_raiz() -> Path:
    for candidato in [Path.cwd(), *Path.cwd().parents]:
        if (candidato / "dados_brutos").is_dir():
            return candidato
    raise FileNotFoundError("Abra o notebook da raiz do projeto ou de 'notebooks/'.")


RAIZ = localizar_raiz()
DIR_BRUTOS = RAIZ / "dados_brutos"
DIR_TRATADOS = RAIZ / "dados_tratados"
DIR_RESULTADOS = RAIZ / "resultados"
for pasta in (DIR_TRATADOS, DIR_RESULTADOS):
    pasta.mkdir(parents=True, exist_ok=True)

SEPARADOR = ";"
ENCODING_SAIDA = "utf-8-sig"
ANOS = (2023, 2024, 2025)
print(RAIZ)


/home/user/PRF-pandas-e-Plotly---2023-2025


In [2]:
def ler_csv_prf(caminho: Path) -> pd.DataFrame:
    """Lê o CSV tentando vários encodings; só falha se todos falharem."""
    for encoding in ("utf-8", "latin1", "utf-8-sig", "cp1252"):
        try:
            df = pd.read_csv(caminho, sep=SEPARADOR, encoding=encoding,
                             low_memory=False, dtype={"id": "string"})
        except (UnicodeDecodeError, UnicodeError):
            continue
        print(f"{caminho.name}: encoding={encoding}, {len(df):,} linhas")
        return df
    raise RuntimeError(f"Nenhum encoding funcionou para {caminho}")


def normalizar_nome(nome: str) -> str:
    nome = str(nome).strip().lower()
    nome = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode()
    return nome.replace(" ", "_").replace("-", "_").replace("/", "_").strip("_")


In [3]:
dados = {}
for ano in ANOS:
    df = ler_csv_prf(DIR_BRUTOS / f"datatran{ano}.csv")
    df.columns = [normalizar_nome(c) for c in df.columns]
    df["arquivo_origem"] = ano
    dados[ano] = df

df = pd.concat([dados[a] for a in ANOS], ignore_index=True)


NUMERICAS = ["br", "km", "pessoas", "mortos", "feridos", "feridos_leves",
             "feridos_graves", "ilesos", "ignorados", "veiculos"]
for c in NUMERICAS:
    df[c] = pd.to_numeric(
        df[c].astype("string").str.replace(",", ".", regex=False), errors="coerce")
df["data_inversa"] = pd.to_datetime(df["data_inversa"], format="%Y-%m-%d", errors="coerce")

for c in df.columns:
    if str(df[c].dtype) in ("object", "string", "str"):
        df[c] = df[c].astype("string").str.strip().str.replace(r"\s{2,}", " ", regex=True)

# strip + upper nos identificadores (já vêm em caixa alta da fonte); as
# descritivas mantêm a caixa publicada pela PRF.
for c in ("uf", "br", "municipio"):
    if c in df.columns and str(df[c].dtype) in ("object", "string", "str"):
        df[c] = df[c].astype("string").str.upper()

DOMINIOS = {
    "uf": {"AC","AL","AM","AP","BA","CE","DF","ES","GO","MA","MG","MS",
           "MT","PA","PB","PE","PI","PR","RJ","RN","RO","RR","RS","SC",
           "SE","SP","TO"},
    "fase_dia": {"Amanhecer", "Anoitecer", "Plena Noite", "Pleno dia"},
    "tipo_pista": {"Dupla", "Múltipla", "Simples"},
    "uso_solo": {"Não", "Sim"},
    "sentido_via": {"Crescente", "Decrescente", "Não Informado"},
    "classificacao_acidente": {"Com Vítimas Fatais", "Com Vítimas Feridas", "Sem Vítimas"},
    "dia_semana": {"domingo", "segunda-feira", "terça-feira", "quarta-feira",
                   "quinta-feira", "sexta-feira", "sábado"},
}
for coluna, dominio in DOMINIOS.items():
    df.loc[df[coluna].isna() | ~df[coluna].isin(dominio), coluna] = "IGNORADO"

df = df.drop(columns=[c for c in ("latitude", "longitude", "regional",
                                  "delegacia", "uop") if c in df.columns])
df = df.drop_duplicates()

# variáveis da Parte 3
df["acidente_fatal"] = (df["mortos"] >= 1).astype(int)
df["ano_acidente"] = df["data_inversa"].dt.year
df["mes_acidente"] = df["data_inversa"].dt.month
df["fim_de_semana"] = df["data_inversa"].dt.day_of_week.isin([5, 6]).astype(int)

CARNAVAL = {2023: ("2023-02-18", "2023-02-22"), 2024: ("2024-02-10", "2024-02-14"),
            2025: ("2025-03-01", "2025-03-05")}
mes, dia = df["data_inversa"].dt.month, df["data_inversa"].dt.day
fim_de_ano = ((mes == 12) & (dia >= 20)) | ((mes == 1) & (dia <= 2))
carnaval = pd.Series(False, index=df.index)
for ano, (ini, fim) in CARNAVAL.items():
    carnaval |= (df["ano_acidente"] == ano) & df["data_inversa"].between(ini, fim)
df["data_comemorativa"] = np.select(
    [fim_de_ano, carnaval], ["Fim de Ano", "Carnaval"], default="Normal")

hora = pd.to_datetime(df["horario"].astype("string").str.strip(),
                      format="%H:%M:%S", errors="coerce").dt.hour
df["faixa_horaria"] = (hora // 3 * 3).map(lambda h: f"{h:02d}h-{h + 3:02d}h").astype("string")

df_enriquecido = df
print(df_enriquecido.shape)


datatran2023.csv: encoding=latin1, 67,766 linhas


datatran2024.csv: encoding=latin1, 73,202 linhas


datatran2025.csv: encoding=latin1, 72,529 linhas


(213497, 32)


## 4.1 `base_analitica` — cópia completa


In [4]:
base_analitica = df_enriquecido.copy()
print(f"base_analitica: {base_analitica.shape}")
print(sorted(base_analitica.columns))


base_analitica: (213497, 32)
['acidente_fatal', 'ano_acidente', 'arquivo_origem', 'br', 'causa_acidente', 'classificacao_acidente', 'condicao_metereologica', 'data_comemorativa', 'data_inversa', 'dia_semana', 'faixa_horaria', 'fase_dia', 'feridos', 'feridos_graves', 'feridos_leves', 'fim_de_semana', 'horario', 'id', 'ignorados', 'ilesos', 'km', 'mes_acidente', 'mortos', 'municipio', 'pessoas', 'sentido_via', 'tipo_acidente', 'tipo_pista', 'tracado_via', 'uf', 'uso_solo', 'veiculos']


## 4.2 `base_modelavel` — só o que se sabe antes do acidente

Excluídas as 5 colunas proibidas pelo exercício: `mortos`, `feridos`, `feridos_leves`,
`feridos_graves` e `classificacao_acidente`.


In [5]:
PROIBIDAS = ["mortos", "feridos", "feridos_leves", "feridos_graves",
             "classificacao_acidente"]

base_modelavel = df_enriquecido.drop(columns=PROIBIDAS).copy()
print(f"base_modelavel: {base_modelavel.shape}")
print(sorted(base_modelavel.columns))


base_modelavel: (213497, 27)
['acidente_fatal', 'ano_acidente', 'arquivo_origem', 'br', 'causa_acidente', 'condicao_metereologica', 'data_comemorativa', 'data_inversa', 'dia_semana', 'faixa_horaria', 'fase_dia', 'fim_de_semana', 'horario', 'id', 'ignorados', 'ilesos', 'km', 'mes_acidente', 'municipio', 'pessoas', 'sentido_via', 'tipo_acidente', 'tipo_pista', 'tracado_via', 'uf', 'uso_solo', 'veiculos']


## 4.3 `verificar_data_leakage`

Levanta erro se alguma variável derivada do desfecho estiver na base modelável.


In [6]:
def verificar_data_leakage(base: pd.DataFrame, proibidas: list) -> str:
    presentes = [c for c in proibidas if c in base.columns]
    if presentes:
        raise ValueError(f"Data leakage: {presentes}")
    return "OK — nenhuma variável proibida na base modelável."


print(verificar_data_leakage(base_modelavel, PROIBIDAS))


OK — nenhuma variável proibida na base modelável.


## 4.4 Exportação e reabertura com `assert`

CSV com separador `;` e `utf-8-sig`. As bases são reabertas e conferidas contra o original.


In [7]:
ARQ_ANALITICA = DIR_TRATADOS / "base_analitica_prf_2023_2025.csv"
ARQ_MODELAVEL = DIR_TRATADOS / "base_modelavel_prf_2023_2025.csv"

base_analitica.to_csv(ARQ_ANALITICA, index=False, sep=SEPARADOR, encoding=ENCODING_SAIDA)
base_modelavel.to_csv(ARQ_MODELAVEL, index=False, sep=SEPARADOR, encoding=ENCODING_SAIDA)

print(f"{ARQ_ANALITICA.name}: {ARQ_ANALITICA.stat().st_size / 1e6:.1f} MB")
print(f"{ARQ_MODELAVEL.name}: {ARQ_MODELAVEL.stat().st_size / 1e6:.1f} MB")


base_analitica_prf_2023_2025.csv: 51.5 MB
base_modelavel_prf_2023_2025.csv: 45.6 MB


In [8]:
analitica_reaberta = pd.read_csv(ARQ_ANALITICA, sep=SEPARADOR, encoding=ENCODING_SAIDA)
modelavel_reaberta = pd.read_csv(ARQ_MODELAVEL, sep=SEPARADOR, encoding=ENCODING_SAIDA)

assert len(analitica_reaberta) == len(base_analitica), "Linhas divergentes na analítica"
assert len(modelavel_reaberta) == len(base_modelavel), "Linhas divergentes na modelável"
assert list(analitica_reaberta.columns) == list(base_analitica.columns)
assert list(modelavel_reaberta.columns) == list(base_modelavel.columns)
assert (analitica_reaberta["acidente_fatal"] == base_analitica["acidente_fatal"]).all()
assert (modelavel_reaberta["acidente_fatal"] == base_modelavel["acidente_fatal"]).all()

print(f"analítica reaberta: {analitica_reaberta.shape}")
print(f"modelável reaberta: {modelavel_reaberta.shape}")
print("Bases reabertas idênticas ao que foi escrito.")


analítica reaberta: (213497, 32)
modelável reaberta: (213497, 27)
Bases reabertas idênticas ao que foi escrito.


## 4.5 Conferência final


In [9]:
print(verificar_data_leakage(modelavel_reaberta, PROIBIDAS))
print(f"\nbase_analitica : {base_analitica.shape[0]:,} linhas × {base_analitica.shape[1]} colunas")
print(f"base_modelavel : {base_modelavel.shape[0]:,} linhas × {base_modelavel.shape[1]} colunas")
print(f"Fatais         : {int(base_modelavel['acidente_fatal'].sum()):,} "
      f"({base_modelavel['acidente_fatal'].mean():.2%})")
print(f"Nulos na modelável: {int(base_modelavel.isna().sum().sum())}")


OK — nenhuma variável proibida na base modelável.

base_analitica : 213,497 linhas × 32 colunas
base_modelavel : 213,497 linhas × 27 colunas
Fatais         : 15,293 (7.16%)
Nulos na modelável: 0
